# M3 – Steps 3 & 4: Fit surrogates and compare them by leave-one-out CV

Goal: build FM(D, T) surrogates from the oracle data and decide which one M1 should use.

**Inputs:** $x_1$ = D and $x_2$ = T, both scaled to [0, 1] over the oracle box (D 0.35–0.80 m, T 8–40 N).
No log transform: D spans only ~2.3× and T ~5×, so plain scaling is enough. The scaling uses the oracle limits,
not the data, so it stays the same when `result_2.csv` / `result_3.csv` are added.
Disk loading DL = T/(πD²/4) is still used below to *interpret* the results (the exploration showed FM depends
mainly on DL), but it is not a model input.

**Models**
1. Quadratic response surface in (D, T) — baseline
2. RBF interpolant (thin-plate spline and cubic; with and without smoothing)
3. Kriging / Gaussian process with a nugget and one length scale per input (ARD)

The fitting code lives in `src/m3_surrogate/` so M1 and `run_all.py` can reuse it.
For every section: write your **prediction before running**, then your **observation** after.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

# Repo root = the folder that contains src/ (works from notebooks/ or from the repo root)
ROOT = Path.cwd()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from m3_surrogate import (load_oracle, make_features, disk_loading,
                          QuadraticSurrogate, RBFSurrogate, GPSurrogate,
                          loo_predictions, loo_table,
                          D_MIN, D_MAX, T_MIN, T_MAX)

ORACLE_DIR = ROOT / 'data' / 'oracle'      # every result_*.csv in here is used
FIG_DIR = ROOT / 'figures'; FIG_DIR.mkdir(exist_ok=True)
RES_DIR = ROOT / 'results'; RES_DIR.mkdir(exist_ok=True)
pd.set_option('display.width', 140)

## 1. Data and surrogate inputs

In [ ]:
df = load_oracle(ORACLE_DIR)
X = make_features(df.D_m, df.T_N)          # columns: [scaled D, scaled T]
y = df.FM.values
n = len(y)
df['x1_D'] = X[:, 0]
df['x2_T'] = X[:, 1]
print(f"{n} points from {df.batch.nunique()} file(s): {sorted(df.batch.unique())}")
print(df.round(4).to_string())
print(f"\nstd of FM = {y.std(ddof=1):.4f}   (an RMSE close to this means the model learned nothing)")

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5))
sc = ax.scatter(X[:, 0], X[:, 1], c=y, s=120, cmap='viridis', edgecolor='k')
for i, (a, b) in zip(df.index, X):
    ax.annotate(str(i), (a, b), xytext=(5, 5), textcoords='offset points', fontsize=9)
plt.colorbar(sc, label='FM')
ax.set_xlabel('x1 = scaled D'); ax.set_ylabel('x2 = scaled T')
ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
ax.set_title('Oracle points in surrogate-input space')
ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

Observation (how well is the square covered? which corners/edges have no points?): _..._

## 2. Fit all models on all points

The GP's fitted hyperparameters are the first hint about noise:
- **nugget** = noise variance the GP thinks the oracle has; **signal variance** = how much FM varies.
- nugget ≪ signal variance (ratio ≲ 1e-3) → the data look nearly noise-free and the GP almost interpolates.
- ratio ~ 0.01–0.1 or more → the GP treats part of the scatter as noise and smooths through it.
- Caution: with 15 points and no repeated (D, T), the nugget cannot separate *noise* from *sharp real
  features* the kernel cannot follow. A repeat point in request 2 is the only direct noise test.
- Length scales (one for D, one for T): small → FM changes fast along that input; large (near the upper
  bound 10) → the input is almost ignored. Since FM depends mainly on T/D², expect both to matter.

Prediction (nugget/signal ratio; which length scale is shorter?): _..._

In [ ]:
models = {
    'Quadratic (D, T)':        QuadraticSurrogate(),
    'RBF thin-plate':          RBFSurrogate('thin_plate_spline', smoothing=0.0),
    'RBF thin-plate, smoothed': RBFSurrogate('thin_plate_spline', smoothing=1e-3),
    'RBF cubic':               RBFSurrogate('cubic', smoothing=0.0),
    'GP (ARD + nugget)':       GPSurrogate(n_restarts=20, seed=0),
}
fitted = {name: m.fit(X, y) for name, m in models.items()}

gp = fitted['GP (ARD + nugget)']
hp = pd.Series(gp.hyperparameters())
print("GP fitted hyperparameters (all data):")
print(hp.to_string(float_format=lambda v: f"{v:.4g}"))
print(f"\nGP kernel: {gp.gp_.kernel_}")

Note on the smoothed RBF: `smoothing=1e-3` is a hand-picked value, not tuned. If you tune it with LOO,
the LOO score of that model becomes optimistic (it was used for selection); say so in the decision log.

Observation (nugget vs signal; which length scale is shorter, D or T?): _..._

## 3. Leave-one-out cross-validation

For each point i: refit the model **without** point i (for the GP, hyperparameters are re-optimised too),
predict point i, record the error. Metrics:
- **LOO RMSE / MAE** – typical error at an unseen point (compare with std of FM)
- **max |error|** and **worst pt** – where the model fails worst
- **LOO Q²** = 1 − SSE/SST: 1 is perfect, 0 is no better than predicting the mean, < 0 is worse
- **mean z²** (GP only) – see section 4

Prediction (ranking of the models; which points will be hardest?): _..._

In [ ]:
loo = {name: loo_predictions(m, X, y) for name, m in models.items()}   # GP refits n times; takes a few seconds
table = loo_table(loo, y)
table.to_csv(RES_DIR / 'm3_loo_table.csv')
table.round(4)

In [ ]:
# Per-point LOO errors (FM - prediction), sorted by disk loading
err = pd.DataFrame({name: y - mu for name, (mu, sd) in loo.items()}, index=df.index)
err.insert(0, 'DL', df.DL.round(1)); err.insert(1, 'FM', df.FM)
err.sort_values('DL').round(4)

In [ ]:
# Parity plots: LOO prediction vs oracle FM (on the diagonal = perfect)
names = list(loo)
fig, axes = plt.subplots(2, 3, figsize=(13, 8.5), sharex=True, sharey=True)
from matplotlib.colors import LogNorm
lo, hi = y.min() - 0.05, y.max() + 0.05
for ax, name in zip(axes.flat, names):
    mu, sd = loo[name]
    if np.all(np.isfinite(sd)):
        ax.errorbar(y, mu, yerr=2 * sd, fmt='none', ecolor='grey', alpha=0.6, label='±2σ (LOO)')
    sc = ax.scatter(y, mu, c=df.DL, norm=LogNorm(), cmap='plasma', edgecolor='k', s=55, zorder=3)
    for i in df.index:
        ax.annotate(str(i), (y[i-1], mu[i-1]), xytext=(4, 3), textcoords='offset points', fontsize=8)
    ax.plot([lo, hi], [lo, hi], 'k--', lw=1)
    r = table.loc[name]
    ax.set_title(f"{name}\nLOO RMSE={r['LOO RMSE']:.4f}  Q²={r['LOO Q2']:.2f}", fontsize=10)
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi); ax.grid(alpha=0.3)
for ax in axes[-1]: ax.set_xlabel('Oracle FM')
for ax in axes[:, 0]: ax.set_ylabel('LOO prediction')
for ax in axes.flat[len(names):]: ax.set_visible(False)      # hide unused panels
fig.colorbar(sc, ax=axes, label='disk loading DL (N/m²)', shrink=0.8)
fig.suptitle('M3 verification: leave-one-out parity plots')
fig.savefig(FIG_DIR / 'm3_loo_parity.png', dpi=150, bbox_inches='tight')
plt.show()

Observation (ranking; which points are off the diagonal for every model, and where are they in DL?
are the worst points the same as the flagged ones, pts 3 and 15, from the exploration?): _..._

## 4. Is the GP's uncertainty honest? Standardised LOO residuals

$z_i = (\text{FM}_i - \mu_{-i}) / \sigma_{-i}$ where $\mu_{-i}, \sigma_{-i}$ come from the GP fitted without point i
(σ includes the nugget, i.e. it is the std of a new oracle value).
- If the GP's uncertainty is right, z behaves like N(0, 1): about 95 % inside ±2, **mean z² ≈ 1**.
- mean z² ≫ 1 → GP is over-confident (errors bigger than it claims); ≪ 1 → too cautious.
- A few |z| > 3 at the same kind of location → a feature the smooth kernel cannot follow (e.g. a sharp drop),
  rather than random noise.

Prediction: _..._

In [ ]:
mu_gp, sd_gp = loo['GP (ARD + nugget)']
z = (y - mu_gp) / sd_gp
df['z_GP'] = z

fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))
ax = axes[0]
ax.scatter(df.DL, z, c=df.D_m, cmap='coolwarm', edgecolor='k', s=70, zorder=3)
for i in df.index:
    ax.annotate(str(i), (df.DL[i], z[i-1]), xytext=(4, 3), textcoords='offset points', fontsize=8)
for k in (-2, 2): ax.axhline(k, color='r', ls='--', lw=1)
ax.axhline(0, color='grey')
ax.set_xscale('log'); ax.set_xlabel('DL (N/m², log)'); ax.set_ylabel('standardised LOO residual z')
ax.set_title('GP: z vs disk loading (colour = D)'); ax.grid(alpha=0.3, which='both')

ax = axes[1]
zs = np.sort(z)
q = norm.ppf((np.arange(1, n + 1) - 0.5) / n)          # expected N(0,1) quantiles
ax.plot(q, zs, 'o'); ax.plot([-3, 3], [-3, 3], 'k--', lw=1)
ax.set_xlabel('N(0,1) quantile'); ax.set_ylabel('sorted z'); ax.set_title('Normal Q-Q plot of z')
ax.grid(alpha=0.3)
plt.tight_layout(); fig.savefig(FIG_DIR / 'm3_gp_standardised_residuals.png', dpi=150); plt.show()

print(f"mean z² = {np.mean(z**2):.2f}    fraction |z|<2 = {np.mean(np.abs(z) < 2):.2f}")
print(df.loc[np.abs(z) > 2, ['D_m', 'T_N', 'DL', 'FM', 'z_GP']].round(3).to_string())

Observation (is the GP over- or under-confident? are the large |z| points random, or all in one DL region?): _..._

## 5. Plot the surrogates with uncertainty

**5a. FM vs thrust T**, at three diameters. Grey band = GP mean ± 2σ (trend uncertainty, no nugget).
Red circles = oracle points with D close to the slice. Top axis shows the disk loading at that (D, T).

Prediction (where will the GP band be widest? where will the models disagree most?): _..._

In [ ]:
D_slices = [0.40, 0.575, 0.75]
show = ['Quadratic (D, T)', 'RBF thin-plate', 'RBF cubic', 'GP (ARD + nugget)']
T_line = np.linspace(T_MIN, T_MAX, 300)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), sharey=True)
for ax, Ds in zip(axes, D_slices):
    D_line = np.full_like(T_line, Ds)
    Xs = make_features(D_line, T_line)
    mu, sd = fitted['GP (ARD + nugget)'].predict(Xs, return_std=True)
    ax.fill_between(T_line, mu - 2*sd, mu + 2*sd, color='grey', alpha=0.3, label='GP ±2σ')
    for name in show:
        ax.plot(T_line, fitted[name].predict(Xs), lw=1.6, label=name)
    near = np.abs(df.D_m - Ds) < 0.08
    ax.scatter(df.T_N[near], df.FM[near], facecolor='none', edgecolor='r', s=110, zorder=5,
               label='oracle, |D−slice|<0.08')
    for i in df.index[near]:
        ax.annotate(str(i), (df.T_N[i], df.FM[i]), xytext=(6, 4), textcoords='offset points', fontsize=8)
    top = ax.secondary_xaxis('top', functions=(lambda t, d=Ds: t / (np.pi * d**2 / 4),
                                               lambda dl, d=Ds: dl * np.pi * d**2 / 4))
    top.set_xlabel('DL (N/m²)', fontsize=8)
    ax.set_title(f'D = {Ds} m'); ax.set_xlabel('T (N)'); ax.grid(alpha=0.3)
axes[0].set_ylabel('FM'); axes[0].set_ylim(0.3, 0.85)
axes[-1].legend(fontsize=8, loc='lower left')
plt.tight_layout(); fig.savefig(FIG_DIR / 'm3_surrogates_vs_T.png', dpi=150); plt.show()

Observation (does FM peak and then drop as T rises at small D? do the models agree away from the red points?
does the RBF overshoot between points?): _..._

**5b. GP mean and standard deviation over the (D, T) box**, with the oracle points and dashed iso-DL lines (for interpretation).
The σ map shows where the next oracle points would reduce uncertainty most (input for request 2).

In [ ]:
Dg, Tg = np.meshgrid(np.linspace(D_MIN, D_MAX, 120), np.linspace(T_MIN, T_MAX, 120))
mu_g, sd_g = fitted['GP (ARD + nugget)'].predict(make_features(Dg.ravel(), Tg.ravel()), return_std=True)
mu_g, sd_g = mu_g.reshape(Dg.shape), sd_g.reshape(Dg.shape)
DLg = disk_loading(Dg, Tg)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, Z, title, cmap in [(axes[0], mu_g, 'GP mean FM', 'viridis'),
                           (axes[1], sd_g, 'GP std σ (trend)', 'magma')]:
    cf = ax.contourf(Dg, Tg, Z, levels=20, cmap=cmap)
    plt.colorbar(cf, ax=ax)
    cs = ax.contour(Dg, Tg, DLg, levels=[30, 60, 100, 150, 217, 300], colors='w', linewidths=0.8, linestyles='--')
    ax.clabel(cs, fmt='DL=%d', fontsize=8)
    ax.scatter(df.D_m, df.T_N, c='w', edgecolor='k', s=50, zorder=3)
    for i in df.index:
        ax.annotate(str(i), (df.D_m[i], df.T_N[i]), xytext=(4, 4), textcoords='offset points', fontsize=8, color='w')
    ax.set_xlabel('D (m)'); ax.set_ylabel('T (N)'); ax.set_title(title)
plt.tight_layout(); fig.savefig(FIG_DIR / 'm3_gp_mean_std_DT.png', dpi=150); plt.show()

k = np.unravel_index(np.argmax(sd_g), sd_g.shape)
print(f"largest GP σ = {sd_g[k]:.4f} at D = {Dg[k]:.3f} m, T = {Tg[k]:.1f} N, DL = {DLg[k]:.0f} N/m²")

Observation (where is σ largest — corners, the high-DL region (small D, large T), between points? does the mean follow the iso-DL lines?): _..._

## 6. Choice of surrogate for M1 (fill in after running)

Write these in your own words; they go to the formulation card, verification appendix and decision log.

1. LOOCV ranking and the error table (attach `results/m3_loo_table.csv` and `figures/m3_loo_parity.png`): _..._
2. Is the oracle noisy? Evidence: GP nugget vs signal variance, mean z², and that no point is repeated yet: _..._
3. How the chosen model treats noise (interpolate exactly vs nugget/smoothing): _..._
4. What a smooth model handles badly (region in (D, T) / DL, points involved) and what you will do about it: _..._
5. Chosen surrogate for M1, and why: _..._
6. Consequences for request 2 (gap, high-DL drop, a repeat for noise, M1 operating points): _..._